# FinTech × AI × Middle Income Trap — Analysis Notebook

Loads the merged country-year panel built by `data_acquisition_colab.ipynb`,
constructs the F (FinTech) and G (AI) composite indices, defines the
middle-income cohort, runs diagnostics, and estimates the headline
specifications.

**Companion to:** `data_acquisition_colab.ipynb` (acquisition) and
`equations.pdf` (theory).

**Outputs land in:** `/content/drive/MyDrive/fintech_ai_mit/analysis/`

**This version:** F is now constructed from all five Findex waves
(2011, 2014, 2017, 2021, 2024) with carry-forward to fill intermediate years,
so F has within-country variation for the panel fixed-effects estimator.


## 1. Setup and load panel

In [ ]:
!pip install -q linearmodels statsmodels country_converter pyarrow

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

ROOT      = Path("/content/drive/MyDrive/fintech_ai_mit")
RAW       = ROOT/"data/raw"
PROCESSED = ROOT/"data/processed"
ANALYSIS  = ROOT/"analysis"
ANALYSIS.mkdir(parents=True, exist_ok=True)

# Load panel — try parquet first, fall back to CSV
panel_pq  = PROCESSED/"panel.parquet"
panel_csv = PROCESSED/"panel.csv"
if panel_pq.exists():
    panel = pd.read_parquet(panel_pq)
elif panel_csv.exists():
    panel = pd.read_csv(panel_csv)
else:
    raise FileNotFoundError(f"No panel found in {PROCESSED}. Run data acquisition first.")

print(f"Panel shape: {panel.shape}")
print(f"Years:    {panel['year'].min()} – {panel['year'].max()}")
print(f"Countries: {panel['iso3'].nunique()}")

In [ ]:
# Coverage by variable
print("Coverage by variable (% non-null):")
cov = (panel.notna().mean().sort_values(ascending=False) * 100).round(1)
print(cov.to_string())

## 2. Index construction

### 2a. FinTech index F — multi-wave (2011, 2014, 2017, 2021, 2024)

Four-component composite using the World Bank Findex 2025 release (which
bundles all five waves in a single file):

- **digital_payment** = `fin17a_17a1_d` — Made or received a digital payment
- **account** = `account_t_d` — Account at FI or mobile money
- **borrow_formal** = `fin22a_22a1_22g_d` — Borrowed from FI or mobile money
- **save_formal** = `fin10` — Saved at a financial institution

Each component is z-scored using **pooled** mean/sd across all country-years
(preserves both cross-country and over-time variation). The composite F is the
simple NaN-tolerant average of the four z-scores. The 2022 supplemental survey
(176 partial rows, post-COVID) is dropped.


In [ ]:
findex_path = RAW/"findex_all_waves_raw.csv"
if not findex_path.exists():
    raise FileNotFoundError(f"Multi-wave Findex not found at {findex_path}. "
                            f"Run data acquisition Section 6 first.")

findex_all = pd.read_csv(findex_path, low_memory=False)
findex_all.columns = [c.lower().strip() for c in findex_all.columns]
print(f"Findex multi-wave raw: {findex_all.shape}")
print(f"Waves present: {sorted(findex_all['year'].dropna().unique())}")

# Filter to the all-adults rows only (drop demographic sub-groups)
findex_pop = findex_all[findex_all["group"].astype(str).str.lower() == "all"].copy()
print(f"After filtering to group=='all': {len(findex_pop)} rows "
      f"across {findex_pop['year'].nunique()} waves")

# Drop the 2022 supplemental — only 176 rows, post-COVID partial survey
findex_pop = findex_pop[findex_pop["year"] != 2022].copy()
print(f"After dropping 2022: {len(findex_pop)} rows")

In [ ]:
# Component mapping (2025 release column names)
F_COMPONENTS = {
    "digital_payment":  "fin17a_17a1_d",
    "account":          "account_t_d",
    "borrow_formal":    "fin22a_22a1_22g_d",
    "save_formal":      "fin10",
}

# Verify columns exist
missing = [c for c in F_COMPONENTS.values() if c not in findex_pop.columns]
if missing:
    raise RuntimeError(f"Missing F-component columns: {missing}. "
                       f"Available 'fin*' columns: "
                       f"{[c for c in findex_pop.columns if c.startswith('fin')][:20]}")

# Coerce to numeric
for label, col in F_COMPONENTS.items():
    findex_pop[col] = pd.to_numeric(findex_pop[col], errors="coerce")

# ISO3 from the file's codewb column (already 3-letter for most countries)
findex_pop["iso3"] = findex_pop["codewb"].astype(str).str.upper().str.strip()

# Fall back to country_converter for non-3-letter codes
import country_converter as coco
import logging
logging.getLogger("country_converter").setLevel(logging.ERROR)
_CC = coco.CountryConverter()

def to_iso3_simple(names):
    s = pd.Series(names).astype(str).str.strip()
    r = _CC.convert(names=s.tolist(), to="ISO3", not_found=None)
    if isinstance(r, str): r = [r]
    return [None if (isinstance(x, list) or x == "not found") else x for x in r]

bad = ~findex_pop["iso3"].str.fullmatch(r"[A-Z]{3}", na=False)
if bad.any():
    findex_pop.loc[bad, "iso3"] = to_iso3_simple(findex_pop.loc[bad, "countrynewwb"])
findex_pop = findex_pop.dropna(subset=["iso3"]).copy()
print(f"After ISO3: {findex_pop.shape}, {findex_pop['iso3'].nunique()} countries")

In [ ]:
# Aggregate to country-year level (one row per (iso3, year))
keep = ["iso3", "year"] + list(F_COMPONENTS.values())
findex_long = (findex_pop[keep]
               .groupby(["iso3", "year"], as_index=False).mean())
print(f"Country-year panel: {findex_long.shape}, "
      f"{findex_long['iso3'].nunique()} countries × "
      f"{sorted(findex_long['year'].unique())}")

# Pooled z-score for each component
print(f"\nComponent statistics (pooled across all country-years):")
for label, col in F_COMPONENTS.items():
    mu, sd = findex_long[col].mean(), findex_long[col].std()
    findex_long[f"{label}_z"] = (findex_long[col] - mu) / sd
    nn = findex_long[col].notna().sum()
    print(f"  {label:18s} ({col:25s}): n={nn}, μ={mu:.3f}, σ={sd:.3f}")

# Composite F = NaN-tolerant average of z-scored components
z_cols = [f"{label}_z" for label in F_COMPONENTS]
findex_long["F"] = findex_long[z_cols].mean(axis=1, skipna=True)
findex_long["F_n_components"] = findex_long[z_cols].notna().sum(axis=1)

print(f"\nF index summary:")
print(f"  n countries-years: {findex_long['F'].notna().sum()}")
print(f"  mean F: {findex_long['F'].mean():.3f}")
print(f"  sd F:   {findex_long['F'].std():.3f}")
print(f"\nF by wave:")
print(findex_long.groupby("year")["F"].agg(["count", "mean", "std"]).round(3))

findex_long.to_csv(PROCESSED/"fintech_index_multiwave.csv", index=False)

In [ ]:
# Top/bottom countries in latest wave
latest = findex_long["year"].max()
latest_F = findex_long[findex_long["year"] == latest].dropna(subset=["F"])
print(f"\nTop 10 by F in {latest}:")
print(latest_F.nlargest(10, "F")[["iso3","F","F_n_components"]].to_string(index=False))
print(f"\nBottom 10 by F in {latest}:")
print(latest_F.nsmallest(10, "F")[["iso3","F","F_n_components"]].to_string(index=False))

# Within-country change 2011 → 2024 (shows the within-country variation we now have)
F_wide = findex_long.pivot_table(index="iso3", columns="year", values="F")
if 2011 in F_wide.columns and 2024 in F_wide.columns:
    F_wide["delta_2011_2024"] = F_wide[2024] - F_wide[2011]
    biggest_gainers = F_wide.dropna(subset=["delta_2011_2024"]).nlargest(10, "delta_2011_2024")
    print(f"\nBiggest F gainers, 2011 → 2024:")
    print(biggest_gainers[[2011, 2024, "delta_2011_2024"]].round(3))

In [ ]:
# Expand F to all panel years using carry-forward
# Year y takes F from the most recent wave ≤ y. Years before first wave inherit
# the first wave's value (so 2010 borrows from 2011).
F_wave = findex_long[["iso3", "year", "F"]].rename(columns={"year": "wave_year"})

all_iso3 = panel["iso3"].unique()
all_years = list(range(int(panel["year"].min()), int(panel["year"].max()) + 1))

F_expanded_rows = []
for iso3, g in F_wave.groupby("iso3"):
    waves = g.sort_values("wave_year")
    wave_years = waves["wave_year"].values
    wave_F = waves["F"].values
    for y in all_years:
        idx = np.searchsorted(wave_years, y, side="right") - 1
        if idx < 0:
            f_val = wave_F[0] if len(wave_F) > 0 else np.nan
        else:
            f_val = wave_F[idx]
        F_expanded_rows.append({"iso3": iso3, "year": y, "F": f_val})

F_expanded = pd.DataFrame(F_expanded_rows)
print(f"F expanded to full grid: {F_expanded.shape}, "
      f"non-null = {F_expanded['F'].notna().sum():,}")
print(f"Unique countries with F: {F_expanded.dropna(subset=['F'])['iso3'].nunique()}")

### 2b. AI index G (AIPI + OpenAlex composite)

Two pillars available now:
- **Activity** — log OpenAlex AI publications per million population, z-scored within year
- **Adoption** — IMF AIPI total index (2023 cross-section), z-scored across countries

Pending pillar (added when GARI data arrives):
- **Readiness** — Oxford GARI 2024 total score, z-scored across countries

For 2023 we have both pillars; for other years we only have activity.


In [ ]:
# Activity pillar from OpenAlex
ai_pubs = panel[["iso3", "year", "ai_pubs_count"]].dropna(subset=["ai_pubs_count"]).copy()
ai_pubs["ai_pubs_count"] = pd.to_numeric(ai_pubs["ai_pubs_count"], errors='coerce')

if "pop" in panel.columns:
    pop = panel[["iso3", "year", "pop"]].dropna()
    ai_pubs = ai_pubs.merge(pop, on=["iso3", "year"], how="left")
    ai_pubs["ai_pubs_per_mil"] = ai_pubs["ai_pubs_count"] / ai_pubs["pop"]
    ai_pubs["ai_pubs_per_mil_log"] = np.log1p(ai_pubs["ai_pubs_per_mil"])
else:
    print("WARNING: no pop column, using log(count) without normalization")
    ai_pubs["ai_pubs_per_mil_log"] = np.log1p(ai_pubs["ai_pubs_count"])

ai_pubs["G_activity_z"] = (
    ai_pubs.groupby("year")["ai_pubs_per_mil_log"]
    .transform(lambda x: (x - x.mean()) / x.std())
)
print(f"G_activity_z: {ai_pubs.shape}, coverage {ai_pubs['G_activity_z'].notna().sum()}")

In [ ]:
# Adoption pillar from IMF AIPI (2023 cross-section)
aipi_cols_present = [c for c in panel.columns if c.startswith("aipi_")]
print(f"AIPI columns: {aipi_cols_present}")

aipi_2023 = (panel[panel["year"] == 2023][["iso3"] + aipi_cols_present]
             .dropna(how="all", subset=aipi_cols_present))

if "aipi_total" in aipi_cols_present:
    aipi_2023["G_adoption_raw"] = aipi_2023["aipi_total"]
elif len(aipi_cols_present) > 0:
    aipi_2023["G_adoption_raw"] = aipi_2023[aipi_cols_present].mean(axis=1, skipna=True)
    print("aipi_total missing; using mean of sub-indices as G_adoption")
else:
    aipi_2023["G_adoption_raw"] = np.nan

mu, sd = aipi_2023["G_adoption_raw"].mean(), aipi_2023["G_adoption_raw"].std()
aipi_2023["G_adoption_z"] = (aipi_2023["G_adoption_raw"] - mu) / sd
print(f"G_adoption_z: n={aipi_2023['G_adoption_z'].notna().sum()}, "
      f"μ={aipi_2023['G_adoption_z'].mean():.3f}")

In [ ]:
# Build G index — activity by year, adoption broadcast from 2023 snapshot
G = ai_pubs[["iso3", "year", "G_activity_z"]].copy()
aipi_xs = aipi_2023[["iso3", "G_adoption_z"]].drop_duplicates(subset=["iso3"])
G = G.merge(aipi_xs, on="iso3", how="left")

# Two-pillar composite (NaN-tolerant)
G["G"] = G[["G_activity_z", "G_adoption_z"]].mean(axis=1, skipna=True)
G.to_csv(PROCESSED/"ai_index.csv", index=False)

print(f"G index: {G.shape}, country-years with G={G['G'].notna().sum()}")
print(f"\nTop 10 in 2023 by G:")
print(G[G['year'] == 2023].nlargest(10, "G")
      [["iso3","year","G_activity_z","G_adoption_z","G"]].to_string(index=False))

### 2c. Merge F and G onto the panel

In [ ]:
# Drop any existing F/G columns first
for c in ["F", "G", "G_activity_z", "G_adoption_z", "FG", "I", "H",
          "F_MI", "G_MI", "FG_MI", "FGI"]:
    if c in panel.columns:
        panel = panel.drop(columns=[c])

# Merge F (now country-year, with within-country variation)
panel = panel.merge(F_expanded, on=["iso3", "year"], how="left")

# Merge G
panel = panel.merge(G[["iso3", "year", "G", "G_activity_z", "G_adoption_z"]],
                    on=["iso3", "year"], how="left")

# Core interaction
panel["FG"] = panel["F"] * panel["G"]

# Institutions I = WGI government effectiveness
if "wgi_ge" in panel.columns:
    panel["I"] = panel["wgi_ge"]
elif "government_effectiveness" in panel.columns:
    panel["I"] = panel["government_effectiveness"]
else:
    print("WARNING: no WGI government effectiveness; I will be NaN")
    panel["I"] = np.nan

# Human capital H from PWT
if "human_capital_pwt" in panel.columns:
    panel["H"] = panel["human_capital_pwt"]
else:
    print("WARNING: no human_capital_pwt; H will be NaN")
    panel["H"] = np.nan

# Triple FGI
panel["FGI"] = panel["F"] * panel["G"] * panel["I"]

print(f"Panel after F/G merge: {panel.shape}")
print(f"  F coverage:  {panel['F'].notna().sum():,} / {len(panel):,}")
print(f"  G coverage:  {panel['G'].notna().sum():,} / {len(panel):,}")
print(f"  FG coverage: {panel['FG'].notna().sum():,} / {len(panel):,}")
print(f"  I coverage:  {panel['I'].notna().sum():,} / {len(panel):,}")
print(f"  H coverage:  {panel['H'].notna().sum():,} / {len(panel):,}")

## 3. Middle-income cohort definition

Following Felipe et al. (2012) and the World Bank MIT literature, a country
is **persistently middle-income** if it spent ≥10 consecutive years in the
LM or UM income group between `YEAR_MIN` and `YEAR_MAX` without graduating to H.
The flag is country-level (time-invariant for the sample window).

Variants:
- `MI_strict`: ≥10 consecutive MI years AND never reached H
- `MI_loose`:  longest MI run ≥ 5 years
- `MI_current`: currently classified LM or UM (time-varying)


In [ ]:
hist_path = RAW/"income_groups_hist.csv"
if not hist_path.exists():
    print(f"WARNING: {hist_path} missing — using current classification only")
    panel["MI_strict"] = panel["mi_status"] if "mi_status" in panel.columns else np.nan
    panel["MI_loose"]   = panel["MI_strict"]
    panel["MI_current"] = panel["MI_strict"]
else:
    hist = pd.read_csv(hist_path)
    print(f"Historical income groups: {hist.shape}")

    def classify_country(g):
        g = g.sort_values("year")
        in_mi = g["income_group_hist"].isin(["LM", "UM"]).values
        ever_h = (g["income_group_hist"] == "H").any()
        max_run, cur_run = 0, 0
        for x in in_mi:
            cur_run = cur_run + 1 if x else 0
            max_run = max(max_run, cur_run)
        return pd.Series({
            "max_mi_run": max_run,
            "ever_h": ever_h,
            "ever_mi": in_mi.any(),
        })

    country_stats = hist.groupby("iso3").apply(classify_country).reset_index()
    country_stats["MI_strict"] = ((country_stats["max_mi_run"] >= 10)
                                  & (~country_stats["ever_h"])).astype(int)
    country_stats["MI_loose"]  = (country_stats["max_mi_run"] >= 5).astype(int)
    print(f"\nMI cohort sizes:")
    print(f"  MI_strict (≥10y MI, never H): {country_stats['MI_strict'].sum()} countries")
    print(f"  MI_loose  (≥5y MI):           {country_stats['MI_loose'].sum()} countries")

    # Drop any stale columns before merging
    for c in ["MI_strict", "MI_loose"]:
        if c in panel.columns:
            panel = panel.drop(columns=[c])
    panel = panel.merge(country_stats[["iso3", "MI_strict", "MI_loose"]],
                        on="iso3", how="left")

    if "mi_status_hist" in panel.columns:
        panel["MI_current"] = panel["mi_status_hist"]
    elif "mi_status" in panel.columns:
        panel["MI_current"] = panel["mi_status"]
    else:
        panel["MI_current"] = panel["MI_strict"]

# Build triple-interaction terms
panel["F_MI"]  = panel["F"]  * panel["MI_strict"]
panel["G_MI"]  = panel["G"]  * panel["MI_strict"]
panel["FG_MI"] = panel["FG"] * panel["MI_strict"]

print(f"\nFinal panel: {panel.shape}")

## 4. Sanity diagnostics

Before any regression, verify:
1. F and G have sensible joint distribution
2. MI cohort isn't pathologically small or collinear with F/G
3. Outcome variable has non-trivial variance
4. Correlation matrix doesn't reveal redundant controls


In [ ]:
# 4.1 — Coverage table by variable group
print("=" * 70)
print("COVERAGE BY VARIABLE GROUP")
print("=" * 70)
groups = {
    "Outcome":     ["gdp_pc_growth", "gdp_pc_ppp"],
    "F (FinTech)": ["F", "F_n_components"],
    "G (AI)":      ["G", "G_activity_z", "G_adoption_z"],
    "I (Inst.)":   ["I"],
    "H (HC)":      ["H"],
    "Controls":    ["inflation", "trade_pct_gdp", "investment_pct_gdp",
                    "pop_growth", "secondary_enrollment"],
    "MI cohort":   ["MI_strict", "MI_loose", "MI_current"],
}
for label, cols in groups.items():
    cols_present = [c for c in cols if c in panel.columns]
    if cols_present:
        cov = panel[cols_present].notna().mean() * 100
        print(f"\n{label}:")
        for c, v in cov.items():
            print(f"  {c:30s} {v:6.1f}%")

In [ ]:
# 4.2 — Summary stats by MI cohort
print("=" * 70)
print("MEANS BY MI COHORT (sample: non-null F & G)")
print("=" * 70)
core_vars = ["gdp_pc_growth", "F", "G", "I", "H", "inflation", "trade_pct_gdp"]
core_vars = [c for c in core_vars if c in panel.columns]
sub = panel.dropna(subset=["F", "G"])
print(sub.groupby("MI_strict")[core_vars].mean().T.round(3))

In [ ]:
# 4.3 — Correlation matrix
import matplotlib.pyplot as plt

corr_vars = ["gdp_pc_growth", "F", "G", "FG", "I", "H",
             "inflation", "trade_pct_gdp", "investment_pct_gdp"]
corr_vars = [c for c in corr_vars if c in panel.columns]
corr = panel[corr_vars].corr().round(2)
print("Correlation matrix:")
print(corr)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_vars))); ax.set_xticklabels(corr_vars, rotation=45, ha='right')
ax.set_yticks(range(len(corr_vars))); ax.set_yticklabels(corr_vars)
for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha='center', va='center',
                color='white' if abs(corr.iloc[i,j]) > 0.5 else 'black', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.title("Correlation matrix of key variables")
plt.tight_layout()
plt.savefig(ANALYSIS/"corr_matrix.png", dpi=120)
plt.show()

In [ ]:
# 4.4 — F-vs-G joint distribution + within-country F variation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# F vs G in latest year
latest = panel["year"].max()
xs = panel[panel["year"] == latest].dropna(subset=["F", "G"])
sc = axes[0].scatter(xs["F"], xs["G"], alpha=0.5, s=25,
                     c=xs.get("MI_strict", 0), cmap='RdYlBu_r')
axes[0].axhline(0, color='gray', lw=0.5); axes[0].axvline(0, color='gray', lw=0.5)
axes[0].set_xlabel("F (FinTech)")
axes[0].set_ylabel("G (AI)")
axes[0].set_title(f"F vs G in {latest} (n={len(xs)})\nred = MI_strict")

# Histogram of FG
axes[1].hist(panel["FG"].dropna(), bins=40, edgecolor='black')
axes[1].axvline(0, color='red', lw=1)
axes[1].set_xlabel("F × G")
axes[1].set_title("Joint digital index F×G")

# Within-country F trajectories (sample of 8 countries)
sample_iso = (panel.dropna(subset=["F"])
              .groupby("iso3")["F"].std()
              .sort_values(ascending=False).head(8).index)
for iso in sample_iso:
    d = panel[(panel["iso3"] == iso) & panel["F"].notna()].sort_values("year")
    axes[2].plot(d["year"], d["F"], marker='o', alpha=0.7, label=iso)
axes[2].set_xlabel("Year"); axes[2].set_ylabel("F")
axes[2].set_title("F over time — 8 highest-variation countries")
axes[2].legend(loc='best', fontsize=8)

plt.tight_layout()
plt.savefig(ANALYSIS/"fg_distribution.png", dpi=120)
plt.show()

## 5. First regression run

### Spec 1 — Baseline F + G + F×G

$$\Delta \log y_{it} = \alpha + \beta_1 F_{it} + \beta_2 G_{it} + \beta_3 (F \times G)_{it} + X_{it}'\gamma + \mu_i + \lambda_t + \varepsilon_{it}$$

- Outcome: GDP per capita growth (annual %)
- Country FEs $\mu_i$ + year FEs $\lambda_t$
- Country-clustered standard errors

### Spec 4 — Headline F × G × MI triple interaction

Coefficient on $F \times G \times \mathbb{1}[\text{MI}]$ tests whether the
joint FinTech+AI boost works differently for stuck-in-MIT countries.


In [ ]:
# Build regression frame
controls = ["inflation", "trade_pct_gdp", "investment_pct_gdp",
            "pop_growth", "secondary_enrollment"]
controls_present = [c for c in controls if c in panel.columns]

reg_vars = ["iso3", "year", "gdp_pc_growth", "F", "G", "FG"] + controls_present
df = panel[reg_vars].dropna(subset=["gdp_pc_growth", "F", "G"]).copy()
df = df.dropna(subset=controls_present)
print(f"Spec 1 sample: {len(df):,} country-years, {df['iso3'].nunique()} countries")

df = df.set_index(["iso3", "year"])

In [ ]:
# Spec 1 — F + G + F×G with two-way FEs and country-clustered SEs
X1 = df[["F", "G", "FG"] + controls_present]
y1 = df["gdp_pc_growth"]

mod1 = PanelOLS(y1, X1, entity_effects=True, time_effects=True, drop_absorbed=True)
res1 = mod1.fit(cov_type="clustered", cluster_entity=True)
print("=" * 70)
print("SPEC 1 — Baseline F + G + F×G with country & year FEs")
print("=" * 70)
print(res1.summary)

In [ ]:
# Spec 4 — Headline F × G × MI triple interaction
reg_vars4 = (["iso3","year","gdp_pc_growth","F","G","FG","I",
              "MI_strict","F_MI","G_MI","FG_MI"] + controls_present)
df4 = panel[reg_vars4].dropna(subset=["gdp_pc_growth","F","G","MI_strict"]).copy()
df4 = df4.dropna(subset=controls_present)
df4 = df4.set_index(["iso3","year"])

# MI_strict gets absorbed by country FE (time-invariant); interactions survive
X4 = df4[["F","G","FG","F_MI","G_MI","FG_MI"] + controls_present]
y4 = df4["gdp_pc_growth"]

mod4 = PanelOLS(y4, X4, entity_effects=True, time_effects=True, drop_absorbed=True)
res4 = mod4.fit(cov_type="clustered", cluster_entity=True)
print("=" * 70)
print("SPEC 4 — Headline F × G × MI triple interaction")
print("=" * 70)
print(res4.summary)

In [ ]:
# Save results + compact coefficient table
with open(ANALYSIS/"spec1_baseline.txt", "w") as f:
    f.write(str(res1.summary))
with open(ANALYSIS/"spec4_headline.txt", "w") as f:
    f.write(str(res4.summary))

print("=" * 70)
print("HEADLINE COEFFICIENTS — F×G and F×G×MI")
print("=" * 70)
key_coefs = ["F", "G", "FG", "F_MI", "G_MI", "FG_MI"]
rows = []
for k in key_coefs:
    if k in res4.params.index:
        rows.append({
            "term": k,
            "coef": res4.params[k],
            "se":   res4.std_errors[k],
            "t":    res4.tstats[k],
            "p":    res4.pvalues[k],
        })
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(f"\nN = {res4.nobs:,} | within-R² = {res4.rsquared_within:.3f}")

## 6. What to read into these results

If `FG_MI` is **positive and significant**: evidence consistent with the
paper's headline — joint FinTech+AI helps MI countries escape faster than
it helps non-MI countries.

If `FG_MI` is **null or negative**: the effect may be absent, or sample
restrictions (few MI countries with non-null F+G+controls) limit power.
Next steps in that case:

1. Add GARI 2024 readiness pillar to G (currently 2-pillar; missing the
   government-readiness dimension)
2. Replace 2023-only AIPI broadcast with a time-varying adoption proxy
3. Use Stanford AI Index official panel if available

### Pending specs (next session)

- **Spec 7** — Vulnerability threshold: η₉δ₀ρ(1−H)(1−σ^KIS) > η̃₂ + η̃₅I + η₆Hσ^KIS
- **Spec 9** — Escape frontier estimation: F*(G) = (ḡ − a_G·G) / (a_F + a_FG·G)
- **Spec 12** — Frontier-distance IV
- **Spec 13** — Submarine-cable IV (Hjort-Poulsen)
- **Spec 14** — System GMM (Arellano-Bond) for dynamic panel
- **Spec 15** — PCA robustness on F and G (rule out simple-average sensitivity)
